In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types  import *
import sys
sys.path.append("/Workspace/Users/sivana9908_gmail.com#ext#@sivana9908gmail.onmicrosoft.com/Uber-Eats-End-to-End-_Azure-Data-Engineering-Project")
from src.common.spark_utils import standardize_columns
from delta.tables import DeltaTable

In [0]:
#Reading data from adls 
df_order_items = (
    spark.read
    .format("csv")
    .option("header", "true")
    .option("inferSchema", "true")
    .load("abfss://bronze@ubereaststorage.dfs.core.windows.net/sql/order_items/")
)
display(df_order_items)


In [0]:
# standardize columns
df_menu = standardize_columns(df_order_items)



#handle duplicates
df_menu = df_menu.dropDuplicates(["order_id"])


#validation
duplication_count=df_menu.groupBy("order_id").count().filter(col("count")>1).count()
print("duplicate menu items",duplication_count)











In [0]:
table_name= "ubereats_databricks.silver.silver_order_items"

if not spark.catalog.tableExists(table_name):

     df_order_items .write.format("delta").mode("overwrite").saveAsTable("ubereats_databricks.silver.silver_order_items")   
      
else:

    target = DeltaTable.forName(spark, table_name)
    target.alias("t") \
        .merge(
            df_menu.alias("s"),
            "t.order_id = s.order_id" )\
        .whenMatchedUpdateAll() \
        .whenNotMatchedInsertAll() \
        .execute()
